In [ ]:
import os 
import pandas as pd
import sqlite3
from flask import Flask, jsonify

In [ ]:
df = pd.read_parquet(r"C:\Users\Quantam Waffle\OneDrive\Desktop\Data engineering and management\Data engineering project\team_1.parquet")

In [ ]:
wide_df = df.pivot_table(
    index=[
        "station_id",
        "state",
        "city",
        "station_name",
        "timestamp",
        "datetime"
    ],
    columns="pollutant",
    values="value"
).reset_index()

In [ ]:
conn = sqlite3.connect("widetable.db")

wide_df.to_sql(
    "pollutant_wide",
    conn,
    if_exists="replace",
    index=False
)

In [ ]:
print(wide_df.isnull().sum())

In [ ]:
missing_pct = wide_df.isnull().mean() * 100
print(missing_pct)

In [ ]:
wide_df = wide_df.drop_duplicates()

In [ ]:
pollutant_cols = [
    col for col in wide_df.columns
    if col not in [
        "station_id",
        "state",
        "city",
        "station_name",
        "timestamp",
        "datetime"
    ]
]

for col in pollutant_cols:
    wide_df.loc[wide_df[col] < 0, col] = None

In [ ]:
for col in pollutant_cols:
    wide_df[col] = wide_df[col].fillna(
        wide_df[col].median()
    )

In [ ]:
app = Flask(__name__)

@app.route("/pollutants")
def pollutants():

    conn = sqlite3.connect("widetable.db")

    df = pd.read_sql(
        "SELECT * FROM pollutant_wide LIMIT 100",
        conn
    )

    return jsonify(
        df.to_dict(orient="records")
    )

app.run(debug=True)